# Visit with Us — Wellness Tourism Package Purchase Prediction


**Objective:** predict `ProdTaken` (0 = no purchase, 1 = purchase) using customer and sales-interaction attributes. This notebook builds a reproducible project repository, validates and prepares the supplied CSV, trains/tunes a model with experiment tracking, creates a Streamlit application, and defines GitHub Actions CI/CD.

## Environment and project configuration

In [2]:
from pathlib import Path
import os, sys, json, subprocess, textwrap, warnings
import numpy as np
import pandas as pd
from IPython.display import display, Markdown

RANDOM_STATE = 42
PROJECT_ROOT = Path("tourism_project")
DATA_DIR = PROJECT_ROOT / "data"
MODEL_BUILDING_DIR = PROJECT_ROOT / "model_building"
DEPLOYMENT_DIR = PROJECT_ROOT / "deployment"
for folder in (DATA_DIR, MODEL_BUILDING_DIR, DEPLOYMENT_DIR, Path(".github/workflows")):
    folder.mkdir(parents=True, exist_ok=True)

# Locate the supplied CSV. In Colab, upload tourism.csv to the session if it is not present.
candidate_paths = [Path("tourism.csv"), Path("visit-with-us-tourism-mlops\tourism_project\data\tourism.csv")]
source_csv = next((p for p in candidate_paths if p.exists()), None)
if source_csv is None:
    try:
        from google.colab import files
        uploaded = files.upload()
        source_csv = Path("tourism.csv")
    except Exception as exc:
        raise FileNotFoundError("Place tourism.csv beside this notebook and rerun this cell.") from exc

import shutil
shutil.copy2(source_csv, DATA_DIR / "tourism.csv")
print("Project folders created.")
print("Dataset copied to:", DATA_DIR / "tourism.csv")

FileNotFoundError: Place tourism.csv beside this notebook and rerun this cell.

## 1. Data Registration

The registration script verifies the expected schema, target availability, duplicate records, and basic data quality. It prints a compact dataset summary and exits with an error if required columns are absent.

In [ ]:
expected_columns = [
    "CustomerID", "ProdTaken", "Age", "TypeofContact", "CityTier",
    "DurationOfPitch", "Occupation", "Gender", "NumberOfPersonVisiting",
    "NumberOfFollowups", "ProductPitched", "PreferredPropertyStar",
    "MaritalStatus", "NumberOfTrips", "Passport", "PitchSatisfactionScore",
    "OwnCar", "NumberOfChildrenVisiting", "Designation", "MonthlyIncome"
]
register_script = r"""
from pathlib import Path
import pandas as pd

DATA_PATH = Path("tourism_project/data/tourism.csv")
EXPECTED = __EXPECTED__
df = pd.read_csv(DATA_PATH)
missing = sorted(set(EXPECTED) - set(df.columns))
if missing:
    raise ValueError(f"Missing required columns: {missing}")
if "ProdTaken" not in df or df["ProdTaken"].isna().any():
    raise ValueError("Target ProdTaken is missing or contains null values.")
if not set(df["ProdTaken"].dropna().unique()).issubset({0, 1}):
    raise ValueError("ProdTaken must contain only 0 and 1.")
print(f"Dataset registered: {DATA_PATH}")
print(f"Rows: {len(df):,} | Columns: {df.shape[1]}")
print(f"Duplicate rows: {df.duplicated().sum()}")
print("Target distribution:")
print(df["ProdTaken"].value_counts(dropna=False).sort_index())
print("Missing values (top):")
print(df.isna().sum().sort_values(ascending=False).head(10))
"""
register_script = register_script.replace("__EXPECTED__", repr(expected_columns))
(MODEL_BUILDING_DIR / "data_register.py").write_text(register_script, encoding="utf-8")

raw_df = pd.read_csv(DATA_DIR / "tourism.csv")
print("Input shape:", raw_df.shape)
print("Columns missing from expected schema:", sorted(set(expected_columns) - set(raw_df.columns)))
print("Duplicate rows:", int(raw_df.duplicated().sum()))
print("Target distribution:")
display(raw_df["ProdTaken"].value_counts().rename(index={0:"No purchase (0)",1:"Purchase (1)"}).to_frame("customers"))
print("Registration script saved:", MODEL_BUILDING_DIR / "data_register.py")

Input shape: (4128, 21)
Columns missing from expected schema: []
Duplicate rows: 0
Target distribution:


,customers
ProdTaken,
No purchase (0),3331
Purchase (1),797


Registration script saved: tourism_project\model_building\data_register.py


## 2. Data Preparation

Cleaning rules:
- Drop the CSV-generated `Unnamed: 0` index column and `CustomerID` identifier.
- Remove exact duplicate rows.
- Validate the binary target and remove rows with a missing target.
- Impute missing numeric values with the median and categorical values with the most frequent value inside the model pipeline (fit on training data only).
- Use a stratified 80/20 train/test split so the target ratio is maintained.
- Save `Xtrain.csv`, `Xtest.csv`, `ytrain.csv`, and `ytest.csv`; GitHub Actions transfers them as an artifact.

In [ ]:
prep_script = r"""
from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split

DATA_PATH = Path("tourism_project/data/tourism.csv")
TARGET = "ProdTaken"
DROP_COLUMNS = ["Unnamed: 0", "CustomerID"]
df = pd.read_csv(DATA_PATH)

# Remove non-feature columns and exact duplicates.
df = df.drop(columns=[c for c in DROP_COLUMNS if c in df.columns], errors="ignore")
df = df.drop_duplicates()
df = df.dropna(subset=[TARGET])
df[TARGET] = pd.to_numeric(df[TARGET], errors="coerce")
df = df[df[TARGET].isin([0, 1])]
df[TARGET] = df[TARGET].astype(int)

X = df.drop(columns=[TARGET])
y = df[TARGET]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

X_train.to_csv("Xtrain.csv", index=False)
X_test.to_csv("Xtest.csv", index=False)
y_train.to_frame(TARGET).to_csv("ytrain.csv", index=False)
y_test.to_frame(TARGET).to_csv("ytest.csv", index=False)
print(f"Prepared rows: {len(df)}")
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print("Train target ratio:", y_train.value_counts(normalize=True).round(3).to_dict())
print("Test target ratio:", y_test.value_counts(normalize=True).round(3).to_dict())
"""
(MODEL_BUILDING_DIR / "prep.py").write_text(prep_script, encoding="utf-8")

# Execute the preparation logic in this notebook to display actual results.
clean_df = raw_df.drop(columns=["Unnamed: 0", "CustomerID"], errors="ignore").drop_duplicates()
clean_df = clean_df.dropna(subset=["ProdTaken"])
clean_df["ProdTaken"] = pd.to_numeric(clean_df["ProdTaken"], errors="coerce")
clean_df = clean_df[clean_df["ProdTaken"].isin([0,1])]
clean_df["ProdTaken"] = clean_df["ProdTaken"].astype(int)
X = clean_df.drop(columns="ProdTaken")
y = clean_df["ProdTaken"]
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)
X_train.to_csv("Xtrain.csv", index=False)
X_test.to_csv("Xtest.csv", index=False)
y_train.to_frame("ProdTaken").to_csv("ytrain.csv", index=False)
y_test.to_frame("ProdTaken").to_csv("ytest.csv", index=False)
print("Cleaned dataset shape:", clean_df.shape)
print("Training features:", X_train.shape, "| Test features:", X_test.shape)
print("Training target proportions:")
display(y_train.value_counts(normalize=True).rename("proportion").to_frame())
print("Testing target proportions:")
display(y_test.value_counts(normalize=True).rename("proportion").to_frame())
print("Saved split files:", [p for p in ["Xtrain.csv","Xtest.csv","ytrain.csv","ytest.csv"] if Path(p).exists()])

Cleaned dataset shape: (4011, 19)
Training features: (3208, 18) | Test features: (803, 18)
Training target proportions:


,proportion
ProdTaken,
0,0.807045
1,0.192955


Testing target proportions:


,proportion
ProdTaken,
0,0.806974
1,0.193026


Saved split files: ['Xtrain.csv', 'Xtest.csv', 'ytrain.csv', 'ytest.csv']


## 3. Model Building and Experimentation Tracking

A `RandomForestClassifier` is selected from the model families permitted by the assignment. The preprocessing pipeline imputes missing values, scales numeric columns, and one-hot encodes categorical columns. `GridSearchCV` tunes the model using stratified cross-validation and F1 scoring, which balances precision and recall for the positive purchase class.

MLflow tracking is enabled when `mlflow` is installed and a tracking server is configured. For a local notebook run without an MLflow server, the script still trains, evaluates, and saves the model; it prints a clear tracking note. GitHub Actions uses a local MLflow file store so experiment metadata is retained as a workflow artifact where configured.

In [ ]:
train_script = r"""
from pathlib import Path
import json
import joblib
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report

X_train = pd.read_csv("Xtrain.csv")
X_test = pd.read_csv("Xtest.csv")
y_train = pd.read_csv("ytrain.csv")["ProdTaken"]
y_test = pd.read_csv("ytest.csv")["ProdTaken"]

numeric = X_train.select_dtypes(include=["number"]).columns.tolist()
categorical = X_train.select_dtypes(exclude=["number"]).columns.tolist()
numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])
preprocessor = ColumnTransformer([
    ("numeric", numeric_pipe, numeric),
    ("categorical", categorical_pipe, categorical)
])
pipeline = Pipeline([
    ("preprocess", preprocessor),
    ("model", RandomForestClassifier(random_state=42, class_weight="balanced", n_jobs=-1))
])
param_grid = {
    "model__n_estimators": [200, 400],
    "model__max_depth": [None, 12],
    "model__min_samples_leaf": [1, 3]
}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
search = GridSearchCV(pipeline, param_grid, scoring="f1", cv=cv, n_jobs=-1, refit=True, verbose=1)
search.fit(X_train, y_train)
best_model = search.best_estimator_
pred = best_model.predict(X_test)
prob = best_model.predict_proba(X_test)[:, 1]
metrics = {
    "accuracy": float(accuracy_score(y_test, pred)),
    "precision": float(precision_score(y_test, pred, zero_division=0)),
    "recall": float(recall_score(y_test, pred, zero_division=0)),
    "f1": float(f1_score(y_test, pred, zero_division=0)),
    "roc_auc": float(roc_auc_score(y_test, prob))
}
print("Best parameters:", search.best_params_)
print("Best CV F1:", round(search.best_score_, 4))
print("Test metrics:", json.dumps(metrics, indent=2))
print(classification_report(y_test, pred, zero_division=0))

# Optional MLflow tracking; model training remains reproducible without an external server.
try:
    import mlflow
    mlflow.set_tracking_uri("file:./mlruns")
    mlflow.set_experiment("visit-with-us-tourism")
    with mlflow.start_run():
        mlflow.log_params(search.best_params_)
        mlflow.log_metric("best_cv_f1", float(search.best_score_))
        mlflow.log_metrics(metrics)
        mlflow.sklearn.log_model(best_model, "model")
    print("MLflow run logged to ./mlruns")
except ImportError:
    print("MLflow is not installed in this environment; install requirements to enable tracking.")
except Exception as exc:
    print(f"MLflow tracking was skipped: {exc}")

Path("tourism_project/deployment").mkdir(parents=True, exist_ok=True)
joblib.dump(best_model, "tourism_project/deployment/tourism_model.joblib")
with open("tourism_project/deployment/model_metrics.json", "w") as f:
    json.dump({"best_params": search.best_params_, "metrics": metrics}, f, indent=2)
print("Saved model: tourism_project/deployment/tourism_model.joblib")
"""
(MODEL_BUILDING_DIR / "train.py").write_text(train_script, encoding="utf-8")

# Execute equivalent training in notebook so real evaluation outputs are visible.
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report

numeric = X_train.select_dtypes(include=["number"]).columns.tolist()
categorical = X_train.select_dtypes(exclude=["number"]).columns.tolist()
preprocessor = ColumnTransformer([
    ("numeric", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]), numeric),
    ("categorical", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("onehot", OneHotEncoder(handle_unknown="ignore"))]), categorical)
])
pipeline = Pipeline([
    ("preprocess", preprocessor),
    ("model", RandomForestClassifier(random_state=RANDOM_STATE, class_weight="balanced", n_jobs=-1))
])
param_grid = {
    "model__n_estimators": [200, 400],
    "model__max_depth": [None, 12],
    "model__min_samples_leaf": [1, 3]
}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
search = GridSearchCV(pipeline, param_grid, scoring="f1", cv=cv, n_jobs=-1, refit=True, verbose=0)
search.fit(X_train, y_train)
best_model = search.best_estimator_
y_pred = best_model.predict(X_test)
y_prob = best_model.predict_proba(X_test)[:,1]
metrics = {
    "accuracy": float(accuracy_score(y_test, y_pred)),
    "precision": float(precision_score(y_test, y_pred, zero_division=0)),
    "recall": float(recall_score(y_test, y_pred, zero_division=0)),
    "f1": float(f1_score(y_test, y_pred, zero_division=0)),
    "roc_auc": float(roc_auc_score(y_test, y_prob))
}
display(pd.DataFrame({"Metric": list(metrics.keys()), "Test score": list(metrics.values())}).round(4))
print("Best parameters:", search.best_params_)
print("Best cross-validation F1:", round(search.best_score_, 4))
print("\\nClassification report:")
print(classification_report(y_test, y_pred, zero_division=0))
import joblib
joblib.dump(best_model, DEPLOYMENT_DIR / "tourism_model.joblib")
(DEPLOYMENT_DIR / "model_metrics.json").write_text(json.dumps({"best_params":search.best_params_, "metrics":metrics}, indent=2))
print("Saved trained model and metrics in", DEPLOYMENT_DIR)

,Metric,Test score
0,accuracy,0.8879
1,precision,0.7444
2,recall,0.6387
3,f1,0.6875
4,roc_auc,0.9215


Best parameters: {'model__max_depth': 12, 'model__min_samples_leaf': 3, 'model__n_estimators': 200}
Best cross-validation F1: 0.6718
\nClassification report:
              precision    recall  f1-score   support

           0       0.92      0.95      0.93       648
           1       0.74      0.64      0.69       155

    accuracy                           0.89       803
   macro avg       0.83      0.79      0.81       803
weighted avg       0.88      0.89      0.88       803

Saved trained model and metrics in tourism_project\deployment


## 4. Model Deployment

The Streamlit app loads the model saved by the training job, collects customer attributes, builds a one-row pandas DataFrame, and displays the predicted purchase class and probability. The app validates values through Streamlit widgets; the preprocessing pipeline handles missing values and unseen categories.

In [ ]:
app_code = r"""
from pathlib import Path
import json
import joblib
import pandas as pd
import streamlit as st

st.set_page_config(page_title="Visit with Us | Wellness Package", page_icon="✈️", layout="wide")
st.title("Visit with Us — Wellness Tourism Package")
st.write("Estimate whether a customer is likely to purchase the Wellness Tourism Package.")

MODEL_PATH = Path(__file__).with_name("tourism_model.joblib")
@st.cache_resource
def load_model():
    return joblib.load(MODEL_PATH)

model = load_model()

with st.form("customer_details"):
    c1, c2, c3 = st.columns(3)
    with c1:
        age = st.number_input("Age", min_value=18, max_value=100, value=35)
        contact = st.selectbox("Type of contact", ["Self Enquiry", "Company Invited"])
        city_tier = st.selectbox("City tier", [1, 2, 3], index=1)
        occupation = st.selectbox("Occupation", ["Salaried", "Small Business", "Large Business", "Free Lancer"])
        gender = st.selectbox("Gender", ["Male", "Female"])
        marital = st.selectbox("Marital status", ["Married", "Single", "Divorced"])
    with c2:
        duration = st.number_input("Duration of pitch (minutes)", min_value=0.0, value=10.0)
        persons = st.number_input("Number of persons visiting", min_value=1, max_value=10, value=2)
        followups = st.number_input("Number of follow-ups", min_value=0, max_value=20, value=3)
        product = st.selectbox("Product pitched", ["Basic", "Standard", "Deluxe", "Super Deluxe", "King"])
        property_star = st.selectbox("Preferred property star", [3, 4, 5], index=0)
        designation = st.selectbox("Designation", ["Executive", "Manager", "Senior Manager", "AVP", "VP"])
    with c3:
        trips = st.number_input("Number of trips per year", min_value=0.0, value=3.0)
        passport = st.selectbox("Has passport?", [0, 1], format_func=lambda x: "Yes" if x else "No")
        pitch_score = st.selectbox("Pitch satisfaction score", [1, 2, 3, 4, 5], index=2)
        own_car = st.selectbox("Owns a car?", [0, 1], format_func=lambda x: "Yes" if x else "No")
        children = st.number_input("Children below age 5 visiting", min_value=0, max_value=10, value=0)
        income = st.number_input("Monthly income", min_value=0.0, value=25000.0)
    submitted = st.form_submit_button("Predict purchase likelihood")

if submitted:
    input_df = pd.DataFrame([{
        "Age": age, "TypeofContact": contact, "CityTier": city_tier,
        "DurationOfPitch": duration, "Occupation": occupation, "Gender": gender,
        "NumberOfPersonVisiting": persons, "NumberOfFollowups": followups,
        "ProductPitched": product, "PreferredPropertyStar": property_star,
        "MaritalStatus": marital, "NumberOfTrips": trips, "Passport": passport,
        "PitchSatisfactionScore": pitch_score, "OwnCar": own_car,
        "NumberOfChildrenVisiting": children, "Designation": designation,
        "MonthlyIncome": income
    }])
    prediction = int(model.predict(input_df)[0])
    probability = float(model.predict_proba(input_df)[0, 1])
    st.subheader("Prediction")
    st.metric("Purchase probability", f"{probability:.1%}")
    if prediction == 1:
        st.success("Likely to purchase — consider prioritizing this customer for follow-up.")
    else:
        st.info("Less likely to purchase — consider a tailored engagement strategy.")
    st.caption("This is a model estimate, not a guarantee of customer behavior.")
"""
(DEPLOYMENT_DIR / "app.py").write_text(app_code, encoding="utf-8")
deployment_requirements = """streamlit==1.41.1
pandas==2.2.3
scikit-learn==1.8.0
joblib==1.4.2
"""
(DEPLOYMENT_DIR / "requirements.txt").write_text(deployment_requirements, encoding="utf-8")
print("Created:", DEPLOYMENT_DIR / "app.py")
print("Created:", DEPLOYMENT_DIR / "requirements.txt")
print("Model exists:", (DEPLOYMENT_DIR / "tourism_model.joblib").exists())

Created: tourism_project\deployment\app.py
Created: tourism_project\deployment\requirements.txt
Model exists: True


## 5. MLOps Pipeline with GitHub Actions

The workflow below has three dependent jobs:
1. **register-dataset** validates the CSV and uploads it as an artifact.
2. **data-prep** downloads the registered dataset, creates stratified splits, and uploads the split files.
3. **model-training** downloads the split artifact, tunes/evaluates the model, logs to a local MLflow store, and commits the refreshed model and metrics to `main`.

The workflow is triggered on pushes to `main` and manually through `workflow_dispatch`. The model commit uses `[skip ci]` to avoid a self-triggering loop.

In [ ]:
workflow = r"""
name: Tourism Project Pipeline

on:
  push:
    branches: [main]
    paths-ignore:
      - "tourism_project/deployment/tourism_model.joblib"
      - "tourism_project/deployment/model_metrics.json"
  workflow_dispatch:

permissions:
  contents: write

jobs:
  register-dataset:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: "3.11"
          cache: "pip"
      - name: Install dependencies
        run: python -m pip install --upgrade pip pandas
      - name: Register dataset
        run: python tourism_project/model_building/data_register.py
      - name: Upload registered dataset
        uses: actions/upload-artifact@v4
        with:
          name: registered-data
          path: tourism_project/data/tourism.csv

  data-prep:
    needs: register-dataset
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: "3.11"
          cache: "pip"
      - name: Install dependencies
        run: python -m pip install pandas scikit-learn==1.8.0
      - name: Download registered dataset
        uses: actions/download-artifact@v4
        with:
          name: registered-data
          path: tourism_project/data
      - name: Run data preparation
        run: python tourism_project/model_building/prep.py
      - name: Upload train-test splits
        uses: actions/upload-artifact@v4
        with:
          name: data-splits
          path: |
            Xtrain.csv
            Xtest.csv
            ytrain.csv
            ytest.csv

  model-training:
    needs: data-prep
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: "3.11"
          cache: "pip"
      - name: Install dependencies
        run: python -m pip install pandas scikit-learn==1.8.0 joblib mlflow==3.0.1
      - name: Download train-test splits
        uses: actions/download-artifact@v4
        with:
          name: data-splits
      - name: Train, tune, evaluate and track model
        env:
          MLFLOW_TRACKING_URI: file:./mlruns
        run: python tourism_project/model_building/train.py
      - name: Upload MLflow tracking data
        if: always()
        uses: actions/upload-artifact@v4
        with:
          name: mlflow-runs
          path: mlruns/
          if-no-files-found: ignore
      - name: Commit trained model and metrics
        run: |
          git config user.name "github-actions[bot]"
          git config user.email "41898282+github-actions[bot]@users.noreply.github.com"
          git add tourism_project/deployment/tourism_model.joblib tourism_project/deployment/model_metrics.json
          git commit -m "Update trained tourism model [skip ci]" || echo "No model changes to commit"
          git push
"""
Path(".github/workflows/pipeline.yml").write_text(workflow, encoding="utf-8")
root_requirements = """pandas==2.2.3
numpy==1.26.4
scikit-learn==1.8.0
joblib==1.4.2
mlflow==3.0.1
streamlit==1.41.1
PyGithub==2.6.1
"""
(PROJECT_ROOT / "requirements.txt").write_text(root_requirements, encoding="utf-8")
print("Workflow created:", Path(".github/workflows/pipeline.yml"))
print("Root requirements created:", PROJECT_ROOT / "requirements.txt")
print("Workflow jobs:", ["register-dataset", "data-prep", "model-training"])

Workflow created: .github\workflows\pipeline.yml
Root requirements created: tourism_project\requirements.txt
Workflow jobs: ['register-dataset', 'data-prep', 'model-training']


## 6. Output Evaluation

https://github.com/02vjiya/visit-with-us-tourism-mlops

https://visit-with-us-tourism-mlops-kf7iwqhwl4cznjqyz236um.streamlit.app/

Evidence-Proof -> https://github.com/02vjiya/visit-with-us-tourism-mlops/evidence

Project Infor -> https://github.com/02vjiya/visit-with-us-tourism-mlops/README.md
